# 01 — Data cleaning & target construction

**What Makes a Song Sticky?** This notebook loads Spotify track-level audio features and prepares analysis-ready tables.

**Framing (important):** We are *not* measuring addiction, replays, skips, or saves. Public datasets rarely expose individual listening behavior. Here, **popularity** is used as an **observable proxy** for songs that are broadly *sticky* or replayable in aggregate. Findings are associative, not causal.


In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

# Project root: works if cwd is project root or notebooks/
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from src import data_prep

RAW_PATH = PROJECT_ROOT / "data" / "raw" / "spotify_tracks.csv"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
CLEAN_PATH = PROCESSED_DIR / "spotify_cleaned.csv"
MODEL_PATH = PROCESSED_DIR / "spotify_model_data.csv"

print("Project root:", PROJECT_ROOT)


## 1. Load dataset


In [ ]:
df = data_prep.load_data(RAW_PATH)


## 2. Inspect shape, columns, sample rows


In [ ]:
print("Shape:", df.shape)
print("Columns:", list(df.columns))
display(df.head())
df.info()


## 3. Missing value analysis


In [ ]:
missing = df.isna().sum().sort_values(ascending=False)
missing_pct = (missing / len(df) * 100).round(2)
miss_df = pd.DataFrame({"missing_count": missing, "pct": missing_pct})
miss_df = miss_df[miss_df["missing_count"] > 0]
print("Columns with missing values:")
display(miss_df if len(miss_df) else "(none)")


## 4. Column normalization & duplicate handling


In [ ]:
df = data_prep.map_expected_columns(df)
print("After mapping, columns:", list(df.columns))

df, n_dup = data_prep.remove_duplicates(df)
print(f"Removed {n_dup} duplicate rows")
print("Shape after dedup:", df.shape)


## 5. Handle missing values (core modeling columns)


In [ ]:
df, missing_before = data_prep.handle_missing_values(df)
print("Missing counts before row drop (selected columns):")
print(missing_before[missing_before > 0])
print("Shape after dropping rows with missing core fields:", df.shape)


## 6. Basic validation (ranges)


In [ ]:
df, val_notes = data_prep.validate_ranges(df)
print("Validation notes:", val_notes)

# Drop rows where invalid 0–1 features were nanned
core = [c for c in data_prep.MODEL_NUMERIC_FEATURES if c in df.columns]
df = df.dropna(subset=[c for c in core if c in df.columns]).reset_index(drop=True)
print("Shape after dropping rows with invalid/nan core features:", df.shape)


## 7. Summary statistics


In [ ]:
num_cols = df.select_dtypes(include=[np.number]).columns
display(df[num_cols].describe().T)


## 8. Standardized popularity & sticky label (top 20%)


In [ ]:
df = data_prep.create_standardized_popularity(df, col="popularity")
df, sticky_threshold = data_prep.create_sticky_label(df, popularity_col="popularity", percentile=0.8)

print(f"Sticky threshold (80th percentile of popularity): {sticky_threshold:.4f}")
print(df["sticky"].value_counts())
print("Class balance (proportion sticky):", df["sticky"].mean().round(4))


## 9. Select columns & save processed datasets


In [ ]:
keep_meta = [c for c in ("track_name", "artist_name", "genre") if c in df.columns]
keep_num = [c for c in data_prep.EXPECTED_COLUMNS if c in df.columns and c not in keep_meta]
extra = ["popularity_z", "sticky"]
all_cols = keep_meta + [c for c in keep_num if c not in keep_meta] + [e for e in extra if e in df.columns]
df_out = df[all_cols].copy()

data_prep.save_processed_data(df_out, CLEAN_PATH)
print("Saved:", CLEAN_PATH)

# Modeling table: same as cleaned for this pipeline (explicit artifact for downstream notebooks)
data_prep.save_processed_data(df_out, MODEL_PATH)
print("Saved:", MODEL_PATH)


## 10. Notes

- **`sticky`**: 1 if popularity is at or above the 80th percentile (top ~20%), else 0.
- **`popularity_z`**: z-score of popularity within this dataset (useful for optional regression-style summaries).

Next: run `02_eda.ipynb` for exploratory visuals comparing sticky vs non-sticky songs.
